In [ ]:
# PRIVATE CELL
username = 'MarcelloCeresini'
repository = 'QuestionAnswering'

In [ ]:
# COLAB ONLY CELLS
try:
    import google.colab
    IN_COLAB = True
    !pip3 install transformers
    !nvidia-smi             # Check which GPU has been chosen for us
    !rm -rf logs
    #from google.colab import drive
    #drive.mount('/content/drive')
    #%cd /content/drive/MyDrive/GitHub/
    !git clone https://github.com/{username}/{repository}.git
    %cd {repository}/src
    %ls
except:
    IN_COLAB = False

### Imports

In [ ]:
# %load_ext tensorboard
import datetime

import os
from tqdm import tqdm
import random

from typing import List, Dict, Tuple

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision

import gc

%matplotlib inline

In [ ]:
import wandb
from wandb.keras import WandbCallback
wandb.login()

In [ ]:
from config import Config
config = Config()
import utils

# Fix random seed for reproducibility
np.random.seed(config.RANDOM_SEED)
random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)

In [ ]:
using_TPU = False

if using_TPU:
    try: 
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='')
        tf.config.experimental_connect_to_cluster(resolver)
        # This is the TPU initialization code that has to be at the beginning.
        tf.tpu.experimental.initialize_tpu_system(resolver)
        print("All devices: ", tf.config.list_logical_devices('TPU'))
        strategy = tf.distribute.TPUStrategy(resolver)
    except:
        print("TPUs are not available, set flag 'using_TPU' to False.")
else:
    physical_devices = tf.config.list_physical_devices('GPU')
    print("Num GPUs Available: ", len(physical_devices))
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
    except:
        # Invalid device or cannot modify virtual devices once initialized.
        pass
    os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
    mixed_precision.set_global_policy('mixed_float16')

In [ ]:
train_ds = tf.data.experimental.load(config.SAVE_PATH_TRAIN_DS_TRAINING_NER)
val_ds = tf.data.experimental.load(config.SAVE_PATH_VAL_DS_TRAINING_NER)

In [ ]:
sweep_config = {}

In [ ]:
sweep_id = wandb.sweep(sweep_config)
wandb.agent(sweep_id, train_function, count=5)

In [ ]:
def train_NER():
    